# Normalize Combined Output

Flattens `combined_output.xlsx` into a single sheet with consistent columns:
- **Person** — the commission recipient (from `Opportunity Owner` or `Mapped_Name`)
- **Commission** — the amount (from `Final Commission Adjusted` or `Managerial Commission`)
- **Commission Type** — `Individual` or `Managerial`

All other detail columns are preserved. The output is safe to hand to any agent or tool — a simple `SUM(Commission)` gives the correct grand total with no double-counting.

In [ ]:
# ── Configuration ──────────────────────────────────────────────
INPUT_FILE  = "combined_output.xlsx"
OUTPUT_FILE = "combined_normalized.xlsx"

In [ ]:
from pathlib import Path
import pandas as pd

INDIVIDUAL_VALUE = "Final Commission Adjusted"
MANAGERIAL_VALUE = "Managerial Commission"

excel = pd.ExcelFile(INPUT_FILE)
records = []

for sheet in excel.sheet_names:
    df = pd.read_excel(excel, sheet_name=sheet)

    if MANAGERIAL_VALUE in df.columns:
        df = df.rename(columns={
            "Mapped_Name": "Person",
            MANAGERIAL_VALUE: "Commission",
        })
        df["Commission Type"] = "Managerial"
    elif INDIVIDUAL_VALUE in df.columns:
        df = df.rename(columns={
            "Opportunity Owner": "Person",
            INDIVIDUAL_VALUE: "Commission",
        })
        df["Commission Type"] = "Individual"
    else:
        print(f"  Skipping sheet '{sheet}' (no commission column)")
        continue

    df["Source Sheet"] = sheet
    records.append(df)
    print(f"  ✓ {sheet} → {len(df)} rows ({df['Commission Type'].iloc[0]})")

combined = pd.concat(records, ignore_index=True)

# Reorder so the key columns come first
cols = ["Person", "Commission", "Commission Type", "Source Sheet"]
rest = [c for c in combined.columns if c not in cols]
combined = combined[cols + rest]

print(f"\n{len(combined)} total rows, {combined['Person'].nunique()} people")
print(f"Grand total: ${combined['Commission'].sum():,.2f}")

In [ ]:
combined.to_excel(OUTPUT_FILE, index=False, sheet_name="Normalized")
print(f"Saved {OUTPUT_FILE}")